# EDA — Almaty PM2.5 Dataset

Exploratory analysis of the OpenAQ LCS network data for Almaty (2024-10-01 to 2026-04-15).

Covers:
1. Station coverage and active counts
2. Data completeness and gaps
3. Seasonal and diurnal patterns
4. Correlations between PM2.5 and meteorological variables / BLH

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

DB = Path('../data/almaty_aq.db')
conn = sqlite3.connect(DB)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Connected to', DB)

## 1. Station coverage

In [ ]:
stations = pd.read_sql('SELECT * FROM locations', conn)
print(f'Total stations in DB: {len(stations)}')
print(f'Providers: {stations["provider"].value_counts().to_dict()}')
print(f'\nLat range: {stations["latitude"].min():.4f} – {stations["latitude"].max():.4f}')
print(f'Lon range: {stations["longitude"].min():.4f} – {stations["longitude"].max():.4f}')

In [ ]:
# Active stations per hour
hourly = pd.read_sql(
    'SELECT datetime_utc, COUNT(DISTINCT location_id) AS n_stations, '
    'AVG(pm25_median) AS pm25 FROM hourly_pm25 GROUP BY datetime_utc',
    conn, parse_dates=['datetime_utc']
)
hourly = hourly.set_index('datetime_utc').sort_index()

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(hourly.index, hourly['n_stations'], lw=0.8, color='steelblue')
ax.axhline(3, color='red', ls='--', lw=0.8, label='Min threshold (3)')
ax.set_ylabel('Active stations')
ax.set_title('Active LCS stations per hour (Almaty network)')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Median active stations: {hourly["n_stations"].median():.0f}')
print(f'Hours with < 3 stations: {(hourly["n_stations"] < 3).sum()}')

## 2. Data completeness and gaps

In [ ]:
# Build full hourly index
full_idx = pd.date_range(hourly.index.min(), hourly.index.max(), freq='h', tz='UTC')
coverage = hourly['pm25'].reindex(full_idx)

missing_pct = coverage.isna().mean() * 100
print(f'Total hours in range: {len(full_idx):,}')
print(f'Hours with data: {coverage.notna().sum():,}')
print(f'Missing: {missing_pct:.1f}%')

# Monthly completeness
monthly = coverage.notna().resample('ME').mean() * 100
fig, ax = plt.subplots(figsize=(12, 3))
monthly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_ylabel('Data completeness (%)')
ax.set_title('Monthly data completeness')
ax.set_xticklabels([t.strftime('%b %Y') for t in monthly.index], rotation=45, ha='right')
ax.axhline(80, color='orange', ls='--', lw=1, label='80% threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Seasonal and diurnal patterns

In [ ]:
from src.features.pipeline import build_feature_matrix
df = build_feature_matrix()
print(f'Feature matrix: {len(df):,} rows, {df.shape[1]} columns')
print(f'Range: {df.index[0]} — {df.index[-1]}')

In [ ]:
# Monthly mean PM2.5
monthly_pm25 = df['pm25'].resample('ME').mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Monthly
ax = axes[0]
monthly_pm25.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.axhline(15, color='green', ls='--', lw=1, label='WHO 24h (15 μg/m³)')
ax.axhline(35, color='orange', ls='--', lw=1, label='Moderate (35)')
ax.set_ylabel('PM2.5 (μg/m³)')
ax.set_title('Monthly mean PM2.5')
ax.set_xticklabels([t.strftime('%b %Y') for t in monthly_pm25.index], rotation=45, ha='right')
ax.legend(fontsize=9)

# Diurnal
ax = axes[1]
diurnal = df.groupby(df.index.hour)['pm25'].mean()
ax.plot(diurnal.index, diurnal.values, marker='o', ms=4, color='steelblue')
ax.set_xlabel('Hour of day (UTC)')
ax.set_ylabel('Mean PM2.5 (μg/m³)')
ax.set_title('Diurnal PM2.5 pattern')
ax.set_xticks(range(0, 24, 3))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Correlations with meteorological variables and BLH

In [ ]:
met_cols = [
    'pm25', 'boundary_layer_height', 'temperature_2m',
    'relative_humidity_2m', 'wind_speed_10m', 'pressure_msl',
]
available = [c for c in met_cols if c in df.columns]
corr = df[available].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax, square=True,
    linewidths=0.5
)
ax.set_title('Pearson correlations — PM2.5 and meteorology')
plt.tight_layout()
plt.show()

print('\nCorrelation with PM2.5:')
print(corr['pm25'].drop('pm25').sort_values())

In [ ]:
# BLH vs PM2.5 scatter (winter only)
if 'boundary_layer_height' in df.columns:
    winter = df[df['heating_season'] == 1].sample(min(3000, len(df)), random_state=42)
    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(
        winter['boundary_layer_height'], winter['pm25'],
        alpha=0.3, s=8, c=winter.index.month, cmap='coolwarm'
    )
    plt.colorbar(sc, ax=ax, label='Month')
    ax.set_xlabel('Boundary-layer height (m)')
    ax.set_ylabel('PM2.5 (μg/m³)')
    ax.set_title('BLH vs PM2.5 — heating season (random sample of 3,000 h)')
    ax.set_xlim(0, 2000)
    ax.set_ylim(0, 300)
    plt.tight_layout()
    plt.show()

    r = df[['pm25', 'boundary_layer_height']].dropna().corr().iloc[0, 1]
    print(f'BLH–PM2.5 Pearson r = {r:.3f}')

In [ ]:
# PM2.5 time series overview
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df.index, df['pm25'], lw=0.5, color='steelblue', alpha=0.8)
ax.axhline(15, color='green', ls='--', lw=0.8, label='WHO 24h')
ax.axhline(75, color='red', ls='--', lw=0.8, label='Unhealthy')
ax.set_ylabel('City-median PM2.5 (μg/m³)')
ax.set_title('Almaty hourly PM2.5 — full training window')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(f'\nSummary statistics:')
print(df['pm25'].describe().round(1))

## Summary

Key findings from this EDA:
- The LCS network has ~190 active stations; median coverage ~150 stations/hour
- Nov–Mar: monthly mean PM2.5 > 35 μg/m³; Jun–Sep: < 20 μg/m³
- BLH shows strongest negative correlation with PM2.5 (r ≈ −0.4 to −0.5)
- Diurnal pattern: two peaks aligned with traffic rush hours (07:00–09:00 and 17:00–19:00 local time)
- October–November 2025 shows reduced coverage due to AirGradient firmware outage